In [1]:
import os
from pathlib import Path

# 1. Install required packages
!pip install streamlit pyngrok pandas numpy scikit-learn matplotlib folium streamlit-folium -q

# 2. Set your Ngrok authtoken (Replace with your actual token)
os.environ["NGROK_AUTHTOKEN"] = "3I2z1ETAyul36lKt2DNcg8pI1aC_2PntmGXfwK5HtcaP2jqb2"

# 3. Verify CSV file presence
csv_path = Path("/content/pune_real_estate_300.csv")

if csv_path.exists():
    print(f"✅ Success! File '{csv_path.name}' is visible and ready.")
    print(f"   File size: {csv_path.stat().st_size / 1024:.2f} KB")
else:
    print(f"❌ Error: File '{csv_path.name}' NOT found in /content/")
    print("   Please upload 'pune_real_estate_300.csv' using the left sidebar Files menu before running Step 1.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 105.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.9/536.9 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 116.0 MB/s eta 0:00:00
✅ Success! File 'pune_real_estate_300.csv' is visible and ready.
   File size: 80.02 KB


In [4]:
%%writefile /content/db.py
import sqlite3
import pandas as pd
from pathlib import Path

DB_PATH = Path("/content/real_estate.db")
INQUIRIES_CSV = Path("/content/user_inquiries.csv")

def get_connection():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

def init_db(csv_path="/content/pune_real_estate_300.csv"):
    conn = get_connection()
    cursor = conn.cursor()

    if Path(csv_path).exists():
        df = pd.read_csv(csv_path)
        df.to_sql("properties", conn, if_exists="replace", index=False)

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS user_inquiries (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            property_id TEXT,
            user_name TEXT,
            user_email TEXT,
            user_phone TEXT,
            message TEXT,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)
    conn.commit()
    conn.close()

def load_properties_from_db():
    conn = get_connection()
    df = pd.read_sql_query("SELECT * FROM properties", conn)
    conn.close()
    return df

def add_inquiry(property_id, name, email, phone, message):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("""
        INSERT INTO user_inquiries (property_id, user_name, user_email, user_phone, message)
        VALUES (?, ?, ?, ?, ?)
    """, (property_id, name, email, phone, message))
    conn.commit()
    conn.close()

    # Save a copy directly to private Excel/CSV in Colab
    df_inquiries = get_all_inquiries()
    df_inquiries.to_csv(INQUIRIES_CSV, index=False)

def get_all_inquiries():
    conn = get_connection()
    df = pd.read_sql_query("SELECT * FROM user_inquiries ORDER BY created_at DESC", conn)
    conn.close()
    return df

def insert_new_property(prop_dict):
    conn = get_connection()
    df = pd.DataFrame([prop_dict])
    df.to_sql("properties", conn, if_exists="append", index=False)
    conn.close()

Writing /content/db.py


In [12]:
%%writefile /content/app.py
import os
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import streamlit as st

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

import db

# Initialize SQLite DBMS
db.init_db("/content/pune_real_estate_300.csv")

try:
    import folium
    from streamlit_folium import st_folium
    MAP_AVAILABLE = True
except Exception:
    MAP_AVAILABLE = False


# ============================================================
# PAGE CONFIG & STYLING
# ============================================================

st.set_page_config(
    page_title="AI Realty | Pune DBMS",
    page_icon="🏠",
    layout="wide",
    initial_sidebar_state="collapsed",
)

# Custom CSS for modern tab headers and primary styling
st.markdown("""
<style>
    .stButton>button {
        background-color: #FF4B4B !important;
        color: white !important;
        border-radius: 6px !important;
        font-weight: 600 !important;
        border: none !important;
    }
    .badge {
        background-color: #E2E8F0;
        color: #4A5568;
        padding: 4px 10px;
        border-radius: 12px;
        font-size: 11px;
        font-weight: bold;
        margin-right: 6px;
    }
    /* Tab Styling */
    .stTabs [data-baseweb="tab-list"] {
        gap: 12px;
    }
    .stTabs [data-baseweb="tab"] {
        height: 45px;
        white-space: pre;
        background-color: #f1f5f9;
        border-radius: 6px 6px 0px 0px;
        padding: 8px 16px;
        font-weight: 600;
    }
    .stTabs [aria-selected="true"] {
        background-color: #FF4B4B !important;
        color: white !important;
    }
</style>
""", unsafe_allow_html=True)

RATES = {
    "stamp_male": 0.07,
    "stamp_female": 0.06,
    "stamp_joint": 0.065,
    "registration_rate": 0.01,
    "registration_cap": 30000.0,
}

LOCALITY_COORDS = {
    "Baner": (18.5590, 73.7868), "Wakad": (18.5986, 73.7636),
    "Hinjewadi": (18.5912, 73.7389), "Bavdhan": (18.5186, 73.7780),
    "Kothrud": (18.5074, 73.8077), "Koregaon Park": (18.5362, 73.8939),
    "Shivajinagar": (18.5308, 73.8470), "Wagholi": (18.5793, 73.9908),
    "Pimpri": (18.6298, 73.7997), "Katraj": (18.4529, 73.8652),
    "Mulshi": (18.5111, 73.5060), "Pirangut": (18.5074, 73.6820),
}

DOCUMENT_TYPES = [
    "Occupancy Certificate (OC)", "Commencement Certificate (CC)", "7/12 Extract / Property Card",
    "Index-II", "Previous Sale Deed / Title Chain", "Encumbrance Certificate",
    "Sanctioned Building Plan", "Fire NOC", "Environment NOC", "AAI Height Clearance"
]

def money(value):
    if value is None or pd.isna(value): return "₹0"
    val = float(value)
    if abs(val) >= 1e7: return f"₹{val / 1e7:.2f} Cr"
    if abs(val) >= 1e5: return f"₹{val / 1e5:.2f} L"
    if abs(val) >= 1e3: return f"₹{val / 1e3:.1f}K"
    return f"₹{val:,.0f}"


# ============================================================
# DATA ENGINE WITH FULL SCORE DISTRIBUTION (40 - 95+)
# ============================================================

@st.cache_data
def load_data():
    df = db.load_properties_from_db()

    numeric_cols = [
        "BHK", "AreaSqFt", "Bathrooms", "ParkingSpaces", "AgeYears",
        "MetroDistanceKm", "ITParkDistanceKm", "MallDistanceKm", "SchoolDistanceKm",
        "HospitalDistanceKm", "Price2022", "CurrentPrice", "DemandIndex", "FutureGrowthScore",
        "RentalDemandScore", "ExpectedShortTermGrowthPct"
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    df["PricePerSqFt"] = np.where(df["AreaSqFt"] > 0, df["CurrentPrice"] / df["AreaSqFt"], 0)
    df["HistoricalGrowthPct"] = np.where(
        df["Price2022"] > 0,
        ((df["CurrentPrice"] - df["Price2022"]) / df["Price2022"]) * 100,
        0,
    )

    features = [
        "Location", "PropertyType", "ListingType", "BHK", "AreaSqFt",
        "Bathrooms", "ParkingSpaces", "AgeYears", "MetroDistanceKm",
        "ITParkDistanceKm", "MallDistanceKm", "SchoolDistanceKm",
        "HospitalDistanceKm", "Furnishing", "Price2022", "DemandIndex",
        "FutureGrowthScore", "RentalDemandScore", "ExpectedShortTermGrowthPct"
    ]

    X_train, X_test, y_train, y_test = train_test_split(
        df[features], df["CurrentPrice"], test_size=0.20, random_state=42
    )
    preprocessor = ColumnTransformer(
        transformers=[("cat", OneHotEncoder(handle_unknown="ignore"), ["Location", "PropertyType", "ListingType", "Furnishing"])],
        remainder="passthrough",
    )
    model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("rf", RandomForestRegressor(n_estimators=100, random_state=42)),
    ])
    model.fit(X_train, y_train)

    df["AI_EstimatedValue"] = model.predict(df[features])
    df["PriceDifferencePct"] = np.where(
        df["AI_EstimatedValue"] != 0,
        ((df["CurrentPrice"] - df["AI_EstimatedValue"]) / df["AI_EstimatedValue"]) * 100,
        0,
    )

    # Dynamic scoring equation to distribute scores across 40s, 50s, 60s, 70s, 80s, and 90s
    norm_growth = (df["FutureGrowthScore"] - df["FutureGrowthScore"].min()) / (df["FutureGrowthScore"].max() - df["FutureGrowthScore"].min() + 1e-5)
    norm_demand = (df["DemandIndex"] - df["DemandIndex"].min()) / (df["DemandIndex"].max() - df["DemandIndex"].min() + 1e-5)
    norm_age = 1 - ((df["AgeYears"] - df["AgeYears"].min()) / (df["AgeYears"].max() - df["AgeYears"].min() + 1e-5))

    # Combined score formula spanning 40 to 98
    raw_score = 40 + (norm_growth * 25) + (norm_demand * 20) + (norm_age * 15) - (df["PriceDifferencePct"] * 0.3)
    df["InvestmentScore"] = np.clip(raw_score, 40.0, 98.5).round(1)

    def recommendation(row):
        if row["InvestmentScore"] >= 80: return "BUY NOW"
        if row["InvestmentScore"] >= 65: return "BUY / HOLD"
        return "NEGOTIATE"

    df["Recommendation"] = df.apply(recommendation, axis=1)
    df = df.sort_values("InvestmentScore", ascending=False).reset_index(drop=True)
    df["PropertyRank"] = np.arange(1, len(df) + 1)

    return df

df = load_data()


# ============================================================
# TOP NAVIGATION TABS
# ============================================================

st.markdown("## 🏠 AI Realty Advisor")
st.caption("Pune Real-Estate Market")

tab_home, tab_search, tab_details, tab_layout, tab_map, tab_costs, tab_market, tab_docs, tab_admin = st.tabs([
    "🏠 Home",
    "🔎 Property Search",
    "📊 Property Details",
    "📐 2D Layout Visualizer",
    "🗺️ Location Intelligence",
    "🧮 Total Cost & EMI",
    "📈 Market Intelligence",
    "📄 Document Verification",
    "🔒 Admin Portal"
])


# ============================================================
# TAB 1: HOME
# ============================================================

with tab_home:
    st.markdown("<span class='badge'>PUNE</span><span class='badge'>AI VALUATION</span><span class='badge'>PROPERTY INTELLIGENCE</span>", unsafe_allow_html=True)
    st.title("Find a property with more clarity.")
    st.caption("Search Pune homes, compare asking price with an AI-estimated value, understand locality factors, check documents and explore an illustrative virtual tour.")

    st.markdown("### What are you looking for?")
    c1, c2, c3 = st.columns(3)
    with c1:
        st.multiselect("Locality", sorted(df["Location"].unique()), placeholder="Choose options")
    with c2:
        st.selectbox("BHK", ["Any", "1", "2", "3", "4", "5"])
    with c3:
        st.number_input("Budget up to (₹)", value=float(df["CurrentPrice"].max()))

    st.button("Search properties", use_container_width=True)

    st.markdown("---")
    st.markdown("### Explore the platform")
    e1, e2, e3, e4 = st.columns(4)
    with e1:
        st.markdown("**👁️ AI Valuation**")
        st.caption("Compare the listed price with the model's estimated property value.")
    with e2:
        st.markdown("**📐 2D + Virtual Tour**")
        st.caption("Explore an illustrative layout and an inside-property walkthrough.")
    with e3:
        st.markdown("**🗺️ Location Intelligence**")
        st.caption("See locality position and available facility-distance data.")
    with e4:
        st.markdown("**📄 Documents & Risk**")
        st.caption("Upload documents and check supplied fields for inconsistencies.")

    st.markdown("---")
    st.markdown("### Featured properties")
    f_cols = st.columns(4)
    top_4 = df.head(4)
    for idx, (_, row) in enumerate(top_4.iterrows()):
        with f_cols[idx]:
            st.caption(f"{row['Location']} • {row['BHK']} BHK")
            st.markdown(f"**{row['PropertyName']}**")
            st.markdown(f"### {money(row['CurrentPrice'])}")
            st.caption(f"AI value {money(row['AI_EstimatedValue'])} • Score {row['InvestmentScore']}")

    st.info("Visualization note: Virtual tours and generated layouts are illustrative. They are not photographs, sanctioned plans, or guaranteed representations of an actual interior.")


# ============================================================
# TAB 2: PROPERTY SEARCH
# ============================================================

with tab_search:
    st.title("Property Search & Inquiry")

    c1, c2 = st.columns(2)
    with c1:
        loc = st.multiselect("Locality Filter", sorted(df["Location"].unique()))
    with c2:
        bhk = st.selectbox("BHK Filter", ["All"] + list(sorted(df["BHK"].unique())))

    filtered = df.copy()
    if loc: filtered = filtered[filtered["Location"].isin(loc)]
    if bhk != "All": filtered = filtered[filtered["BHK"] == bhk]

    st.dataframe(filtered[["PropertyID", "PropertyName", "Location", "BHK", "CurrentPrice", "AI_EstimatedValue", "InvestmentScore"]], use_container_width=True, hide_index=True)

    st.markdown("---")
    st.markdown("### Submit Inquiry")
    with st.form("inquiry_form"):
        p_id = st.selectbox("Property ID", df["PropertyID"].tolist())
        name = st.text_input("Name")
        email = st.text_input("Email")
        phone = st.text_input("Phone Number")
        msg = st.text_area("Message")
        if st.form_submit_button("Submit Inquiry"):
            if name and email:
                db.add_inquiry(p_id, name, email, phone, msg)
                st.success("Inquiry submitted successfully!")
            else:
                st.error("Please enter Name and Email.")


# ============================================================
# TAB 3: PROPERTY DETAILS
# ============================================================

with tab_details:
    st.title("Property profile")
    selected_id = st.selectbox("Select property", df["PropertyID"].tolist(), format_func=lambda x: f"{x} — {df.loc[df['PropertyID'].eq(x), 'PropertyName'].iloc[0]}")
    row = df.loc[df["PropertyID"] == selected_id].iloc[0]

    st.subheader(f"{row['PropertyName']}")
    st.caption(f"{row['Location']} • {row['PropertyType']} • {row['BHK']} BHK • {row['AreaSqFt']} sq ft")

    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Current price", money(row["CurrentPrice"]))
    c2.metric("AI estimated value", money(row["AI_EstimatedValue"]))
    c3.metric("Investment score", f"{row['InvestmentScore']}/100")
    c4.metric("Property rank", f"#{row['PropertyRank']}")

    st.markdown("### Property snapshot")
    snap_data = {
        "Feature": ["Location", "Type", "Listing", "BHK", "Area", "Furnishing", "Age", "Price / sq ft"],
        "Value": [
            row["Location"], row["PropertyType"], row["ListingType"],
            str(row["BHK"]), f"{row['AreaSqFt']} sq ft", row["Furnishing"],
            f"{int(row['AgeYears'])} years", money(row["PricePerSqFt"])
        ]
    }
    st.table(pd.DataFrame(snap_data))


# ============================================================
# TAB 4: 2D LAYOUT VISUALIZER
# ============================================================

with tab_layout:
    st.title("2D floor-plan visualization")
    selected_id = st.selectbox("Select property for layout", df["PropertyID"].tolist())
    row = df.loc[df["PropertyID"] == selected_id].iloc[0]

    st.info("This is an illustrative layout generated from BHK, area, bathroom and balcony attributes. It is not a sanctioned architectural plan.")
    st.button("Generate / refresh layout")

    st.markdown(f"#### Illustrative Layout — {row['PropertyID']} | {row['PropertyType']} | {row['BHK']} BHK | {row['AreaSqFt']} sq ft")

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.set_xlim(0, 16); ax.set_ylim(0, 10); ax.axis("off")

    def room(x, y, w, h, label):
        ax.add_patch(Rectangle((x, y), w, h, fill=False, linewidth=2))
        ax.text(x + w/2, y + h/2, label, ha="center", va="center", fontsize=8, fontweight="bold")

    room(1, 8, 14, 1.5, "ENTRY / PASSAGE")
    room(1, 5.5, 4, 2, "BALCONY")
    room(1, 2, 4, 3, "BEDROOM / FLEX")
    room(5, 2, 3, 3, "DINING / STUDY")
    room(8, 3.5, 2, 1.5, "BATH 1")
    room(1, 0, 5, 2, "LIVING / DINING")
    room(6, 0, 3, 2, "KITCHEN")
    room(9, 0, 3, 2, "BEDROOM 1")
    room(12, 0, 3, 2, "STUDY")

    st.pyplot(fig)


# ============================================================
# TAB 5: LOCATION INTELLIGENCE
# ============================================================

with tab_map:
    st.title("Location intelligence")
    selected_id = st.selectbox("Select property location", df["PropertyID"].tolist())
    row = df.loc[df["PropertyID"] == selected_id].iloc[0]

    st.subheader(f"{row['PropertyName']} — {row['Location']}")
    center = LOCALITY_COORDS.get(row["Location"], (18.5204, 73.8567))

    if MAP_AVAILABLE:
        fmap = folium.Map(location=center, zoom_start=13)
        folium.Marker(center, tooltip=f"{row['PropertyName']}").add_to(fmap)
        st_folium(fmap, width=1000, height=400)

    st.markdown("### Nearby facility distances")
    fac_df = pd.DataFrame({
        "Facility": ["School", "Mall", "Hospital", "Metro", "IT Park"],
        "Distance": [
            f"{float(row['SchoolDistanceKm']):.1f} km",
            f"{float(row['MallDistanceKm']):.1f} km",
            f"{float(row['HospitalDistanceKm']):.1f} km",
            f"{float(row['MetroDistanceKm']):.1f} km",
            f"{float(row['ITParkDistanceKm']):.1f} km"
        ]
    })
    st.table(fac_df)

# ============================================================
# TAB 6: TOTAL COST & EMI
# ============================================================

with tab_costs:
    st.title("Total cost & EMI")
    selected_id = st.selectbox("Select property for valuation", df["PropertyID"].tolist())
    row = df.loc[df["PropertyID"] == selected_id].iloc[0]

    st.info(f"{row['PropertyName']} • {money(row['CurrentPrice'])}")

    c1, c2, c3 = st.columns(3)
    with c1:
        b_type = st.selectbox("Buyer type", ["Male", "Female", "Joint"])
    with c2:
        c_status = st.selectbox("Construction status", ["Ready to move / OC", "Under construction"])
    with c3:
        st.checkbox("Affordable housing category")

    rr_val = st.number_input("Ready Reckoner / applicable value (₹)", value=float(row["CurrentPrice"]))

    stamp = rr_val * RATES["stamp_male"]
    reg = min(rr_val * RATES["registration_rate"], RATES["registration_cap"])
    total_cost = rr_val + stamp + reg

    m1, m2, m3, m4 = st.columns(4)
    m1.metric("Base value", money(rr_val))
    m2.metric("Stamp duty", money(stamp))
    m3.metric("Registration", money(reg))
    m4.metric("GST", "₹0")

    st.success(f"Estimated total acquisition cost: {money(total_cost)}")

    st.markdown("---")
    st.markdown("### 🏦 EMI & affordability")
    e1, e2, e3 = st.columns(3)
    loan = e1.number_input("Loan amount (₹)", value=5000000.0)
    rate = e2.number_input("Interest rate (% p.a.)", value=8.5)
    tenure = e3.number_input("Tenure (years)", value=20)

    r = rate / (12 * 100)
    n = tenure * 12
    emi = (loan * r * (1 + r)**n) / (((1 + r)**n) - 1) if r > 0 else loan / n

    st.metric("Estimated monthly EMI", money(emi))


# ============================================================
# TAB 7: PUNE MARKET INTELLIGENCE
# ============================================================

with tab_market:
    st.title("Pune market intelligence")
    st.caption("Analytics are kept here rather than on the landing page so the product starts simple.")

    k1, k2, k3, k4 = st.columns(4)
    k1.metric("Properties", len(df))
    k2.metric("Localities", df["Location"].nunique())
    k3.metric("Average price", money(df["CurrentPrice"].mean()))
    k4.metric("Average AI value", money(df["AI_EstimatedValue"].mean()))

    st.markdown("---")
    st.markdown("### Highest investment scores")
    st.dataframe(
        df[["PropertyRank", "PropertyID", "PropertyName", "Location", "BHK", "CurrentPrice", "AI_EstimatedValue", "InvestmentScore", "Recommendation"]].head(10),
        use_container_width=True, hide_index=True
    )


# ============================================================
# TAB 8: DOCUMENT VERIFICATION
# ============================================================

with tab_docs:
    st.title("Documents & risk")
    selected_id = st.selectbox("Select property for doc check", df["PropertyID"].tolist())

    st.warning("Document statuses in this prototype are not legal verification. Only independently verified documents should be relied upon for a real transaction.")

    st.markdown("### Document checklist")
    doc_df = pd.DataFrame({
        "Document": DOCUMENT_TYPES,
        "Status": ["Not provided"] * len(DOCUMENT_TYPES)
    })
    st.table(doc_df)

    st.file_uploader("Upload documents (200MB per file • PDF, PNG, JPG, DOC, DOCX)")


# ============================================================
# TAB 9: ADMIN PORTAL & DBMS CRUD + ANALYTICS (ADMIN ONLY)
# ============================================================

with tab_admin:
    st.title("🗄️ Admin Panel & User Submissions")
    st.caption("Inspect live database tables, perform CRUD operations, and view direct SQL analytics.")

    pwd = st.text_input("Enter Admin Password to Access Database Records", type="password")

    if pwd == "admin123":
        sub_tab1, sub_tab2, sub_tab3, sub_tab4 = st.tabs([
            "📩 User Inquiries",
            "📊 Admin Analytics Dashboard",
            "✏️ Manage Listings (CRUD)",
            "📜 Raw Properties Table"
        ])

        # 1. USER INQUIRIES
        with sub_tab1:
            st.subheader("Submitted User Inquiries (user_inquiries SQL Table)")
            inq_df = db.get_all_inquiries()
            if inq_df.empty:
                st.info("No inquiries submitted yet. Go to 'Property Search' tab to submit one!")
            else:
                st.dataframe(inq_df, use_container_width=True, hide_index=True)
                st.download_button(
                    "Download Inquiries CSV",
                    inq_df.to_csv(index=False).encode('utf-8'),
                    "user_inquiries.csv",
                    "text/csv"
                )

        # 2. SQL ANALYTICS DASHBOARD
        with sub_tab2:
            st.subheader("Direct SQL Aggregation Analytics")
            st.caption("Real-time GROUP BY, COUNT, and AVG calculations directly executed on SQLite DBMS.")

            conn = db.get_connection()

            c_a, c_b = st.columns(2)
            with c_a:
                st.markdown("**Inquiries Count by Locality (SQL GROUP BY)**")
                loc_sql = """
                SELECT p.Location, COUNT(*) as InquiryCount, AVG(p.CurrentPrice) as AvgPrice
                FROM user_inquiries i
                JOIN properties p ON i.property_id = p.PropertyID
                GROUP BY p.Location
                ORDER BY InquiryCount DESC
                """
                loc_analytics = pd.read_sql_query(loc_sql, conn)
                if not loc_analytics.empty:
                    st.dataframe(loc_analytics, use_container_width=True, hide_index=True)
                else:
                    st.info("Submit inquiries to populate locality analytics.")

            with c_b:
                st.markdown("**Inquiries by Property Type & Avg Price**")
                type_sql = """
SELECT p.PropertyType, COUNT(i.id) as TotalInquiries, AVG(p.CurrentPrice) as AveragePrice
FROM user_inquiries i
JOIN properties p ON i.property_id = p.PropertyID
GROUP BY p.PropertyType
ORDER BY TotalInquiries DESC
"""
                type_analytics = pd.read_sql_query(type_sql, conn)
                if not type_analytics.empty:
                    st.dataframe(type_analytics, use_container_width=True, hide_index=True)
                else:
                    st.info("Submit inquiries to populate property type analytics.")

            conn.close()

        # 3. FULL CRUD OPERATIONS (CREATE, UPDATE, DELETE)
        with sub_tab3:
            st.subheader("Full Property CRUD Operations")
            crud_action = st.radio("Select Action", ["Create New Property", "Update Existing Property", "Delete / Archive Property"], horizontal=True)

            if crud_action == "Create New Property":
                with st.form("add_prop_crud"):
                    p_id = st.text_input("Property ID", f"P{len(df)+1}")
                    p_name = st.text_input("Property Name")
                    p_loc = st.selectbox("Location", sorted(df["Location"].unique()))
                    p_bhk = st.number_input("BHK", 1, 5, 2)
                    p_price = st.number_input("Current Price (₹)", 1000000.0, 100000000.0, 5000000.0)
                    if st.form_submit_button("Insert Record"):
                        db.insert_new_property({
                            "PropertyID": p_id, "PropertyName": p_name, "Location": p_loc,
                            "BHK": p_bhk, "CurrentPrice": p_price, "AreaSqFt": 1000,
                            "PropertyType": "Apartment", "ListingType": "Sale", "Furnishing": "Semi-Furnished"
                        })
                        st.success("Property inserted into SQLite DBMS!")
                        st.cache_data.clear()

            elif crud_action == "Update Existing Property":
                u_id = st.selectbox("Select Property ID to Edit", df["PropertyID"].tolist())
                p_row = df.loc[df["PropertyID"] == u_id].iloc[0]

                with st.form("update_prop_form"):
                    new_price = st.number_input("Update Price (₹)", value=float(p_row["CurrentPrice"]))
                    new_status = st.selectbox("Listing Status", ["Available", "Under Offer", "Sold", "Draft"])
                    if st.form_submit_button("Update Property"):
                        conn = db.get_connection()
                        cur = conn.cursor()
                        cur.execute("UPDATE properties SET CurrentPrice = ?, ListingType = ? WHERE PropertyID = ?", (new_price, new_status, u_id))
                        conn.commit()
                        conn.close()
                        st.success(f"Property {u_id} updated successfully!")
                        st.cache_data.clear()

            elif crud_action == "Delete / Archive Property":
                d_id = st.selectbox("Select Property ID to Remove", df["PropertyID"].tolist())
                if st.button("Delete Property Permanently"):
                    conn = db.get_connection()
                    cur = conn.cursor()
                    cur.execute("DELETE FROM properties WHERE PropertyID = ?", (d_id,))
                    conn.commit()
                    conn.close()
                    st.warning(f"Property {d_id} removed from database.")
                    st.cache_data.clear()

        # 4. RAW TABLE VIEW
        with sub_tab4:
            st.dataframe(df, use_container_width=True)

    elif pwd != "":
        st.error("Incorrect password.")

Overwriting /content/app.py


In [13]:
import os
import time
from pyngrok import ngrok

!pkill -f streamlit || true
!pkill -f ngrok || true
ngrok.kill()

ngrok.set_auth_token(os.environ["NGROK_AUTHTOKEN"])
!streamlit run /content/app.py --server.address 0.0.0.0 --server.port 8501 > /content/streamlit.log 2>&1 &

time.sleep(5)

try:
    public_url = ngrok.connect(8501)
    print("YOUR WEBSITE IS LIVE AT:")
    print(public_url)
except Exception as e:
    print(f"Error launching URL: {e}")

^C
^C
YOUR WEBSITE IS LIVE AT:
NgrokTunnel: "https://shredder-pasta-knelt.ngrok-free.dev" -> "http://localhost:8501"
